In [19]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── Quadrant divider configuration ────────────────────────────────────────────
QUADRANT_COST_THRESHOLD      = 50.0      # vertical dividing line (x-axis / cost)
QUADRANT_EXPOSURE_THRESHOLD  = 50000.0   # horizontal dividing line (y-axis / exposure)

QUADRANT_LABELS = {
    "bottom_left":  "Healthier and Cheaper",
    "bottom_right": "Healthier but More Expensive",
    "top_left":     "Cheaper but Not as Healthy",
    "top_right":    "Less Healthy and More Expensive",
}
QUADRANT_LINE_COLOR  = "rgba(120, 120, 120, 0.55)"
QUADRANT_LABEL_COLOR = "rgba(100, 100, 100, 0.70)"
QUADRANT_LABEL_SIZE  = 12

# ── Filtered table configuration (runs below both boundaries) ────────────────
RATIO_COLUMN_LABEL = "Cost / Cum Exposure"
RATIO_DECIMALS = 6
# ──────────────────────────────────────────────────────────────────────────────

repo_root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
stats_path = None
for candidate in repo_root_candidates:
    probe = candidate / "Sample Inputs" / "output_childcare_stats.json"
    if probe.exists():
        stats_path = probe
        break

if stats_path is None:
    raise FileNotFoundError("Could not find Sample Inputs/output_childcare_stats.json from the current notebook environment.")

rows = []
with stats_path.open() as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if row.get("event") == "run_summary":
            rows.append(row)

if not rows:
    raise ValueError(f"No run_summary rows found in {stats_path}")

stats_df = pd.DataFrame(rows).sort_values("run_number").reset_index(drop=True)
stats_df["run_duration_minutes"] = stats_df["time"] / 60.0
stats_df["run_duration_hms"] = pd.to_timedelta(stats_df["time"], unit="s").astype(str)
stats_df["run_clock_time"] = pd.to_datetime(stats_df["timestamp"]).dt.strftime("%I:%M%p").str.lower()

hover_columns = {
    "player_name": True,
    "run_number": True,
    "run_duration_hms": True,
    "ach_total_cost": ":.2f",
    "exposure_mean_cumulative": ":.2f",
}

# Cost leaderboard: lowest cost first (rank 1 = lowest)
cost_sorted = stats_df.sort_values("ach_total_cost", ascending=True).reset_index(drop=True)
cost_sorted["leaderboard_label"] = cost_sorted.apply(
    lambda row: f"{row.name + 1}. {row['player_name']} (Run {int(row['run_number'])} - {row['run_clock_time']})",
    axis=1,
)

# Exposure leaderboard: lowest exposure first (rank 1 = lowest/best)
exposure_sorted = stats_df.sort_values("exposure_mean_cumulative", ascending=True).reset_index(drop=True)
exposure_sorted["leaderboard_label"] = exposure_sorted.apply(
    lambda row: f"{row.name + 1}. {row['player_name']} (Run {int(row['run_number'])} - {row['run_clock_time']})",
    axis=1,
)

# Alerts leaderboard: fewest alerts first (rank 1 = lowest/best)
alerts_sorted = stats_df.sort_values("alert_trigger_count", ascending=True).reset_index(drop=True)
alerts_sorted["leaderboard_label"] = alerts_sorted.apply(
    lambda row: f"{row.name + 1}. {row['player_name']} (Run {int(row['run_number'])} - {row['run_clock_time']})",
    axis=1,
)

# Runs in the healthier+cheaper quadrant (below both thresholds)
qualified_df = stats_df[
    (stats_df["ach_total_cost"] < QUADRANT_COST_THRESHOLD)
    & (stats_df["exposure_mean_cumulative"] < QUADRANT_EXPOSURE_THRESHOLD)
].copy()

# Cost-to-exposure ratio; zero exposure yields NA.
qualified_df[RATIO_COLUMN_LABEL] = qualified_df["ach_total_cost"] / qualified_df["exposure_mean_cumulative"].replace(0, pd.NA)
qualified_df = qualified_df.sort_values(["ach_total_cost", "exposure_mean_cumulative"], ascending=[True, True])

cost_fig = px.bar(
    cost_sorted,
    x="ach_total_cost",
    y="leaderboard_label",
    orientation="h",
    hover_data=hover_columns,
    labels={"ach_total_cost": "Total Cost", "leaderboard_label": "Leaderboard"},
)
cost_fig.update_traces(marker_color="#b56576")

exposure_fig = px.bar(
    exposure_sorted,
    x="exposure_mean_cumulative",
    y="leaderboard_label",
    orientation="h",
    hover_data=hover_columns,
    labels={"exposure_mean_cumulative": "Mean Cumulative Exposure", "leaderboard_label": "Leaderboard"},
)
exposure_fig.update_traces(marker_color="#6d597a")

alerts_fig = px.bar(
    alerts_sorted,
    x="alert_trigger_count",
    y="leaderboard_label",
    orientation="h",
    hover_data=hover_columns,
    labels={"alert_trigger_count": "Alert Count", "leaderboard_label": "Leaderboard"},
)
alerts_fig.update_traces(marker_color="#e76f51")

scatter_fig = px.scatter(
    stats_df,
    x="ach_total_cost",
    y="exposure_mean_cumulative",
    hover_data=hover_columns,
    color="player_name",
    text="run_number",
    labels={
        "ach_total_cost": "Total Cost",
        "exposure_mean_cumulative": "Mean Cumulative Exposure",
    },
)
scatter_fig.update_traces(textposition="top center", marker=dict(size=11, opacity=0.9))

if qualified_df.empty:
    table_values = [["No runs below both thresholds."], ["-"], ["-"], ["-"], ["-"]]
else:
    table_values = [
        qualified_df["player_name"].tolist(),
        qualified_df["ach_total_cost"].map(lambda v: f"{v:.2f}").tolist(),
        qualified_df["exposure_mean_cumulative"].map(lambda v: f"{v:.2f}").tolist(),
        qualified_df["alert_trigger_count"].map(lambda v: f"{int(v)}").tolist(),
        qualified_df[RATIO_COLUMN_LABEL].map(
            lambda v: "NA" if pd.isna(v) else f"{float(v):.{RATIO_DECIMALS}f}"
        ).tolist(),
    ]

panel_height = max(300, 20 * len(stats_df) + 160)
table_height = max(220, 34 * max(1, len(qualified_df) + 1))

fig = make_subplots(
    rows=5,
    cols=1,
    specs=[[{"type": "xy"}], [{"type": "xy"}], [{"type": "xy"}], [{"type": "xy"}], [{"type": "table"}]],
    subplot_titles=(
        "Cost Leaderboard (Lowest Cost Wins)",
        "Mean Cumulative Exposure Leaderboard (Lowest Wins)",
        "Alert Trigger Count Leaderboard (Fewest Wins)",
        "Cost vs Mean Cumulative Exposure",
        f"Healthier and Cheaper (Cost < {QUADRANT_COST_THRESHOLD}, Exposure < {QUADRANT_EXPOSURE_THRESHOLD})",
    ),
    vertical_spacing=0.05,
    row_heights=[0.2, 0.2, 0.2, 0.27, 0.13],
)

for trace in cost_fig.data:
    fig.add_trace(trace, row=1, col=1)

for trace in exposure_fig.data:
    fig.add_trace(trace, row=2, col=1)

for trace in alerts_fig.data:
    fig.add_trace(trace, row=3, col=1)

for trace in scatter_fig.data:
    fig.add_trace(trace, row=4, col=1)

fig.add_trace(
    go.Table(
        header=dict(
            values=["Player Name", "Cost", "Cumulative Exposure", "Alert Count", RATIO_COLUMN_LABEL],
            fill_color="#e9ecef",
            align="left",
            font=dict(size=12),
        ),
        cells=dict(
            values=table_values,
            fill_color="white",
            align="left",
            height=28,
            font=dict(size=11),
        ),
    ),
    row=5,
    col=1,
)

fig.update_layout(
    title="Childcare Simulation Run Summary",
    bargap=0.12,
    height=panel_height * 3 + table_height,
    width=900,
    legend_title_text="User",
    hovermode="closest",
    showlegend=True,
)

fig.update_xaxes(title_text="Total Cost", row=1, col=1)
fig.update_yaxes(
    title_text="Leaderboard",
    row=1,
    col=1,
    categoryorder="array",
    categoryarray=cost_sorted["leaderboard_label"].tolist()[::-1],
)
fig.update_xaxes(title_text="Mean Cumulative Exposure", row=2, col=1)
fig.update_yaxes(
    title_text="Leaderboard",
    row=2,
    col=1,
    categoryorder="array",
    categoryarray=exposure_sorted["leaderboard_label"].tolist()[::-1],
)
fig.update_xaxes(title_text="Alert Count", row=3, col=1)
fig.update_yaxes(
    title_text="Leaderboard",
    row=3,
    col=1,
    categoryorder="array",
    categoryarray=alerts_sorted["leaderboard_label"].tolist()[::-1],
)
fig.update_xaxes(title_text="Total Cost", row=4, col=1)
fig.update_yaxes(title_text="Mean Cumulative Exposure", row=4, col=1)

# ── Quadrant lines and labels on panel 4 ──────────────────────────────────────
# Vertical line at QUADRANT_COST_THRESHOLD
fig.add_shape(
    type="line",
    xref="x4",
    yref="y4 domain",
    x0=QUADRANT_COST_THRESHOLD,
    x1=QUADRANT_COST_THRESHOLD,
    y0=0,
    y1=1,
    line=dict(color=QUADRANT_LINE_COLOR, width=1.5, dash="dash"),
)
# Horizontal line at QUADRANT_EXPOSURE_THRESHOLD
fig.add_shape(
    type="line",
    xref="x4 domain",
    yref="y4",
    x0=0,
    x1=1,
    y0=QUADRANT_EXPOSURE_THRESHOLD,
    y1=QUADRANT_EXPOSURE_THRESHOLD,
    line=dict(color=QUADRANT_LINE_COLOR, width=1.5, dash="dash"),
)

# Quadrant label corners: (key, xanchor, x_domain_pos, yanchor, y_domain_pos)
_pad = 0.03
_quadrant_specs = [
    ("bottom_left", "left", _pad, "bottom", _pad),
    ("bottom_right", "right", 1 - _pad, "bottom", _pad),
    ("top_left", "left", _pad, "top", 1 - _pad),
    ("top_right", "right", 1 - _pad, "top", 1 - _pad),
]

for key, xanchor, xpos, yanchor, ypos in _quadrant_specs:
    fig.add_annotation(
        xref="x4 domain",
        yref="y4 domain",
        x=xpos,
        y=ypos,
        text=f"<i>{QUADRANT_LABELS[key]}</i>",
        showarrow=False,
        xanchor=xanchor,
        yanchor=yanchor,
        font=dict(size=QUADRANT_LABEL_SIZE, color=QUADRANT_LABEL_COLOR),
    )
# ──────────────────────────────────────────────────────────────────────────────

# Export standalone interactive HTML for browser viewing.
export_path = stats_path.parent.parent / "analysis" / "game_results_dashboard.html"
fig.write_html(str(export_path), full_html=True, include_plotlyjs=True)
print(f"Saved standalone dashboard to: {export_path}")

fig.show()

stats_df[[
    "player_name",
    "run_number",
    "run_duration_hms",
    "ach_total_cost",
    "exposure_mean_cumulative",
    "exposure_max_cumulative",
    "alert_trigger_count",
    "end_reason",
]].tail(10)

Saved standalone dashboard to: /Users/brylew/github/brave-childcare-godot/analysis/game_results_dashboard.html


,player_name,run_number,run_duration_hms,ach_total_cost,exposure_mean_cumulative,exposure_max_cumulative,alert_trigger_count,end_reason
11,Bryzan,13,0 days 19:59:00.666666114,59.786667,91836.610114,219506.343498,4,schedule_complete
12,Madhav,14,0 days 19:59:01.111110559,59.786667,91930.778544,219570.259507,4,schedule_complete
13,Madhav,15,0 days 19:59:00.444443892,36.570000,86652.857343,198880.019402,5,schedule_complete
14,Andrew,16,0 days 19:59:03.244444474,59.251667,91937.762385,220221.403119,4,schedule_complete
15,Srini,17,0 days 19:55:17.711150367,56.536667,91899.776749,219529.057886,4,manual
16,Srini,18,0 days 19:59:00.444443892,42.865000,86438.094208,201633.342940,5,schedule_complete
17,JohnnyDeuce,19,0 days 19:59:05.777777225,26.570000,147429.798221,373385.611603,4,schedule_complete
18,JohnnyDeuce,20,0 days 19:59:24.133333328,22.173333,239053.510027,504128.613765,4,schedule_complete
19,JohnnyHaleDeuce,21,0 days 19:59:02.888888336,35.650000,86645.490748,198908.366513,5,schedule_complete
20,BillyTrio,22,0 days 19:59:01.999999447,32.560000,160435.689386,351104.931722,4,schedule_complete
